# Data Completeness Report

Analyze extraction completeness across all processed papers:
- Values extracted per paper
- Null/gap analysis
- Data source breakdown (table vs. figure vs. inline text)
- Figure extraction reliability by type

In [ ]:
import json
import sqlite3
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

ROOT_DIR = Path("..").resolve()
EXTRACTIONS_DIR = ROOT_DIR / "data" / "extractions"
DB_PATH = ROOT_DIR / "data" / "sensory_index.db"

In [ ]:
# Load all extracted JSONs
papers = {}
for json_path in sorted(EXTRACTIONS_DIR.glob("*.json")):
    with open(json_path) as f:
        papers[json_path.stem] = json.load(f)

print(f"Loaded {len(papers)} papers")
for sid in papers:
    print(f"  - {sid}")

## 1. Field Completeness per Paper

In [ ]:
def count_completeness(paper):
    """Count populated vs null fields in a paper JSON."""
    total = 0
    populated = 0
    nulls = []
    
    # Metadata fields
    meta = paper.get("study_metadata", {})
    for key, val in meta.items():
        total += 1
        if val is not None and val != "" and val != []:
            populated += 1
        else:
            nulls.append(f"study_metadata.{key}")
    
    # Experiment fields
    for exp in paper.get("experiments", []):
        for section in ["panel", "session_design", "scale"]:
            sec_data = exp.get(section, {})
            if isinstance(sec_data, dict):
                for key, val in sec_data.items():
                    total += 1
                    if val is not None and val != "" and val != []:
                        populated += 1
                    else:
                        nulls.append(f"{section}.{key}")
        
        # Count stimuli
        total += 1
        stimuli = exp.get("stimuli", [])
        if stimuli:
            populated += 1
        else:
            nulls.append("stimuli")
        
        # Count sensory data
        total += 1
        if exp.get("sensory_data"):
            populated += 1
        else:
            nulls.append("sensory_data")
    
    return {"total": total, "populated": populated, "nulls": nulls,
            "completeness": populated / total if total > 0 else 0}

completeness_data = []
for sid, paper in papers.items():
    c = count_completeness(paper)
    c["study_id"] = sid
    completeness_data.append(c)

df_comp = pd.DataFrame(completeness_data)
if not df_comp.empty:
    print(df_comp[["study_id", "total", "populated", "completeness"]].to_string(index=False))

## 2. Data Gaps Analysis

In [ ]:
# Analyze data gaps from extraction_metadata
all_gaps = []
for sid, paper in papers.items():
    gaps = paper.get("extraction_metadata", {}).get("data_gaps", [])
    for gap in gaps:
        gap["study_id"] = sid
        all_gaps.append(gap)

if all_gaps:
    df_gaps = pd.DataFrame(all_gaps)
    print(f"Total data gaps: {len(df_gaps)}")
    print(f"\nGaps by reason:")
    print(df_gaps["reason"].value_counts().to_string())
    print(f"\nGaps by field:")
    print(df_gaps["field"].value_counts().head(10).to_string())
else:
    print("No data gaps recorded (or no papers extracted yet)")

## 3. Data Source Breakdown

In [ ]:
# Analyze where data came from: tables, figures, inline text
def count_data_sources(paper):
    sources = {"table": 0, "figure": 0, "inline_text": 0, "unknown": 0}
    for exp in paper.get("experiments", []):
        sensory = exp.get("sensory_data", {})
        _count_sources_recursive(sensory, sources)
    return sources

def _count_sources_recursive(data, sources, depth=0):
    if depth > 10 or not isinstance(data, dict):
        return
    if "data_source" in data:
        src = str(data["data_source"]).lower()
        if "table" in src:
            sources["table"] += 1
        elif "figure" in src or "fig" in src:
            sources["figure"] += 1
        elif "text" in src or "inline" in src:
            sources["inline_text"] += 1
        else:
            sources["unknown"] += 1
    for val in data.values():
        if isinstance(val, dict):
            _count_sources_recursive(val, sources, depth + 1)

source_data = []
for sid, paper in papers.items():
    s = count_data_sources(paper)
    s["study_id"] = sid
    source_data.append(s)

if source_data:
    df_src = pd.DataFrame(source_data)
    print(df_src.to_string(index=False))
    
    # Plot
    if len(df_src) > 0 and df_src[["table", "figure", "inline_text"]].sum().sum() > 0:
        df_src.set_index("study_id")[["table", "figure", "inline_text"]].plot(
            kind="bar", stacked=True, figsize=(10, 5)
        )
        plt.title("Data Source Breakdown by Paper")
        plt.ylabel("Number of data blocks")
        plt.tight_layout()
        plt.show()

## 4. Figure Extraction Reliability

In [ ]:
# Analyze figure inventory across papers
fig_data = []
for sid, paper in papers.items():
    for fig in paper.get("figure_inventory", []):
        fig["study_id"] = sid
        fig_data.append(fig)

if fig_data:
    df_figs = pd.DataFrame(fig_data)
    print(f"Total figures across all papers: {len(df_figs)}")
    if "status" in df_figs.columns:
        print(f"\nFigure status:")
        print(df_figs["status"].value_counts().to_string())
else:
    print("No figure inventory data available yet")

## 5. SQLite Index Summary

In [ ]:
if DB_PATH.exists():
    conn = sqlite3.connect(DB_PATH)
    df_index = pd.read_sql("SELECT * FROM sensory_index", conn)
    conn.close()
    print(f"Papers in index: {len(df_index)}")
    display(df_index)
else:
    print(f"SQLite index not yet created at {DB_PATH}")